In [ ]:
# modules
#!pip install matplotlib
#!pip install numpy
#!pip install -Uqq fastai
from fastai.vision.all import *
import matplotlib.pyplot as plt
import numpy as np
from functools import partial

from numpy.random import normal,seed, uniform
import pandas as pd
import torch.nn.functional as F

Hello, World! JO


In [ ]:
path = untar_data(URLs.MNIST_SAMPLE)
(path/'train').ls()
threes = (path/'train'/'3').ls().sorted()
sevens = (path/'train'/'7').ls().sorted()
img_path = threes[0]
img = Image.open(img_path)
img
#Saving all three and seven  images in a list in a tensor
threes_imgs = [tensor(Image.open(p)) for p in threes]
sevens_imgs = [tensor(Image.open(p)) for p in sevens]
#checking the length of the lists
len(threes_imgs), len(sevens_imgs)
#stacking the three and seven images into tensors and normalizing them to find their average
threes_stack = torch.stack(threes_imgs).float()/255
sevens_stack = torch.stack(sevens_imgs).float()/255
threes_stack.size()
mean_threes = threes_stack.mean(0)
mean_sevens = sevens_stack.mean(0)
show_image(mean_threes)
show_image(mean_sevens)
#finding the absolute difference between one selected image of three and the mean of its class and perform mean reduction
one_three = threes_stack[0]
diff_one_three_ab = (one_three - mean_threes).abs().mean()
#finding the squared difference between one selected image of three and the mean of its class and perform mean reduction
diff_one_three_sq = ((one_three - mean_threes)**2).mean()
#finding the absolute difference between one selected image of seven and the mean of the three class and perform mean reduction
one_seven = sevens_stack[0]
diff_one_seven_ab = (one_seven - mean_threes).abs().mean()
diff_one_seven_sq = ((one_seven - mean_threes)**2).mean()
print(diff_one_three_ab, diff_one_seven_ab)
print(diff_one_three_sq, diff_one_seven_sq)
if diff_one_three_ab < diff_one_seven_ab:
    print("Predicted: The image is three")
else:
    print("Predicted: The image is seven")
#this abs and squared loss can also be easily implemented by pytorch nn.functional

diff_one_three_ab_fn = F.l1_loss(one_three, mean_threes)
diff_one_three_sq_fn = F.mse_loss(one_three, mean_threes).sqrt()
diff_one_seven_ab_fn = F.l1_loss(one_seven, mean_threes)
diff_one_seven_sq_fn = F.mse_loss(one_seven, mean_threes).sqrt()
print(diff_one_three_ab_fn, diff_one_seven_ab_fn)
print(diff_one_three_sq_fn, diff_one_seven_sq_fn)
#lets incorporate a validation set
valid_threes_stack = torch.stack([tensor(Image.open(p)) for p in (path/'valid'/'3').ls().sorted()]).float()/255
valid_sevens_stack = torch.stack([tensor(Image.open(p)) for p in (path/'valid'/'7').ls().sorted()]).float()/255
valid_sevens_stack.size(), valid_threes_stack.size()
#finding the mean images for the validation set
mean_valid_threes = valid_threes_stack.mean(0)
mean_valid_sevens = valid_sevens_stack.mean(0)
#creating  functions to calculate the absolute and squared difference between an image and the mean of its class
def abs_diff(img,mean_img): return F.l1_loss(img, mean_img)
def sq_diff(img,mean_img): return F.mse_loss(img, mean_img).sqrt()
abs_diff(one_three, mean_threes)
sq_diff(one_three, mean_threes)
abs_diff(one_seven, mean_threes)
sq_diff(one_seven, mean_threes) 
#using pytorch broadcasting to calculate the absolute and squared differences for the entire validation set however it does not work on nn.functional so  it needs new function with broadcasting support
def abs_diff_broadcast(imgs, mean_img): return (imgs - mean_img).abs().mean((-1,-2))
def sq_diff_broadcast(imgs, mean_img): return ((imgs - mean_img)**2).mean((-1,-2))
abs_diff_valid_threes = abs_diff_broadcast(valid_threes_stack, mean_valid_threes)
sq_diff_valid_threes = sq_diff_broadcast(valid_threes_stack, mean_valid_threes)
abs_diff_valid_sevens = abs_diff_broadcast(valid_sevens_stack, mean_valid_sevens)
sq_diff_valid_sevens = sq_diff_broadcast(valid_sevens_stack, mean_valid_sevens)
print(abs_diff_valid_threes.shape, sq_diff_valid_threes.shape)
print(abs_diff_valid_sevens.shape, sq_diff_valid_sevens.shape)
#a function for predicting the class of each image  based on the absolute and squared differences
def is_3_ab(x):
    return abs_diff_broadcast(x, mean_threes) < abs_diff_broadcast(x, mean_sevens)

def is_3_sq(x):
    return sq_diff_broadcast(x, mean_threes) < sq_diff_broadcast(x, mean_sevens)
is_3_ab(one_three), is_3_sq(one_three).float()
# testing the entire validation set using broadcasting on is_3_ab and is_3_sq
is_3_ab(valid_threes_stack)
is_3_sq (valid_threes_stack)
#calculating the accuracy for the validation set based on the absolute and squared differences
accuracy_ab_validation_3_3 = is_3_ab(valid_threes_stack).float().mean()
accuracy_sq_validation_3_3 = is_3_sq(valid_threes_stack).float().mean()
print(accuracy_ab_validation_3_3, accuracy_sq_validation_3_3)

accuracy_ab_validation_7_3 = 1- (is_3_ab(valid_sevens_stack).float()).mean()
accuracy_sq_validation_7_3 = 1-(is_3_sq(valid_sevens_stack).float()).mean()
print(accuracy_ab_validation_7_3, accuracy_sq_validation_7_3)
